In [1]:
import pathlib, pandas as pd, numpy as np
from scipy.special import expit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score


def resolve_path(relative_path: str) -> pathlib.Path:
    """Return an existing Path for the given relative location, trying common Kaggle directories."""
    candidates = [
        pathlib.Path(relative_path),
        pathlib.Path("/kaggle/input") / relative_path,
        pathlib.Path("kaggle/input") / relative_path,
        pathlib.Path("data") / relative_path,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Unable to locate {relative_path}")


base_rel = "jigsaw-toxic-comment-classification-challenge"
train_path = resolve_path(f"{base_rel}/train.csv")
test_path = resolve_path(f"{base_rel}/test.csv")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)




In [2]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df["ex_mark"] = df["comment_text"].str.count("!").clip(0, 1)
    df["qu_mark"] = df["comment_text"].str.count(r"\?").clip(0, 1)
    smileys_good = r"((:|;)-?(\)|P|D))"
    smileys_bad = r"((:|;)-?\'?(\())"
    df["smileys_good"] = (
        df["comment_text"].str.count(smileys_good).clip(0, 1).astype(int)
    )
    df["smileys_bad"] = df["comment_text"].str.count(smileys_bad).clip(0, 1).astype(int)
    return df


df_train = add_features(df_train)
df_test = add_features(df_test)

# Make the vectorizer more restrictive: fewer features and higher min_df
vect = CountVectorizer(
    min_df=800,  # ignore rarer terms more aggressively
    max_features=200,  # smaller vocabulary
    ngram_range=(1, 1),  # unigrams only
    stop_words="english",
)
all_text = pd.concat([df_train["comment_text"], df_test["comment_text"]])
vect.fit(all_text)

X_train_text = vect.transform(df_train["comment_text"])
X_test_text = vect.transform(df_test["comment_text"])

train_features = X_train_text
test_features = X_test_text

Y = df_train[["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]




In [3]:
model = LinearSVC()
# Strengthen regularisation further (smaller C)
params = {"C": [1e-6], "random_state": [0]}

pred_df = pd.DataFrame({"id": df_test["id"]})
scores = []

for col in Y.columns:
    gs = GridSearchCV(model, params, scoring="roc_auc", cv=3, n_jobs=-1)
    gs.fit(train_features, Y[col])
    prob = expit(gs.decision_function(test_features))
    pred_df[col] = prob
    scores.append(gs.best_score_)
    print(f"{col}: {gs.best_score_:.4f}")

print("mean score:", np.mean(scores))




toxic: 0.6908
severe_toxic: 0.7762
obscene: 0.7173
threat: 0.6890
insult: 0.7054
identity_hate: 0.6920
mean score: 0.7117689142460577


In [4]:
submission_path = "submission.csv"
pred_df.to_csv(submission_path, index=False)